# Tuples

**Topic:** `3.2` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Normalise a coordinate tuple

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** tuple rebuilding, rounding, immutability

**Explanation and reasoning**

Unpack, coerce, round, repack - every step produces a new value, so nothing about the original record can change. Validating the arity first means a caller passing a 3-tuple gets a precise error instead of silently ignoring the third field.

In [ ]:
def normalize(point):
    """Return (x, y) rounded to 2 decimals as a new tuple."""
    if len(point) != 2:
        raise ValueError('expected an (x, y) pair')
    x, y = point
    return (round(float(x), 2), round(float(y), 2))

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Integer inputs become floats. Wrong-length tuples raise ValueError.

**Alternative approaches**

A comprehension over the pair is shorter but loses the named x, y clarity.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert normalize((3.14159, 2.71828)) == (3.14, 2.72)
assert normalize((1, 2)) == (1.0, 2.0)
try:
    normalize((1, 2, 3))
    raise AssertionError('should raise')
except ValueError:
    pass

---

## Solution 2 — Swap two records without a temporary variable

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** unpacking, swap, evaluation order

**Explanation and reasoning**

`return b, a` packs the two names into a fresh tuple in one step. The same unpacking trick `a, b = b, a` works in place because Python evaluates the entire right-hand side - producing (b, a) - before binding any name on the left, so nothing is clobbered mid-swap.

In [ ]:
def swap_positions(a, b):
    """Return the pair reversed, without a temporary."""
    return b, a

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Identical values return an equal pair; mixed types are unaffected.

**Alternative approaches**

A temp variable works but adds a line and a name for no benefit.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert swap_positions(('a', 1), ('b', 2)) == (('b', 2), ('a', 1))
assert swap_positions(0, 0) == (0, 0)

---

## Solution 3 — Pack measurement metadata into a validated record

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** record construction, validation, type coercion

**Explanation and reasoning**

Validation runs before coercion so the TypeError and ValueError messages name the caller's actual mistake rather than a float() conversion failure. The bool guard precedes the numeric check for the same reason as in Topic 3.1 - bool subclasses int, and a True slipping through would silently read as 1.0.

In [ ]:
def make_reading(value, unit, timestamp):
    """Validate, coerce, and pack a (float, str, float) record."""
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError('value must be numeric')
    if not unit or not unit.strip():
        raise ValueError('unit must be non-empty')
    if isinstance(timestamp, bool) or not isinstance(timestamp, (int, float)):
        raise TypeError('timestamp must be numeric')
    return (float(value), unit.strip(), float(timestamp))

**Complexity**

Time O(len(unit)); space O(len(unit)).

**Edge cases and failure modes**

Empty and whitespace-only units raise. Negative values are legal.

**Alternative approaches**

A dataclass with __post_init__ scales to more fields but is heavyweight at three.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert make_reading(21, 'C', 1712000000) == (21.0, 'C', 1712000000.0)
try:
    make_reading(1, '', 0)
    raise AssertionError('should raise')
except ValueError:
    pass

---

## Solution 4 — Unpack a heterogeneous log record safely

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** unpacking, star unpacking, sentinel names

**Explanation and reasoning**

One unpacking statement with an explicit `_timestamp` target documents exactly which fields matter; the reader never has to count brackets. The conditional scales only metres, and returning a bare `tag, scaled` packs the promised two-tuple.

In [ ]:
def summarise(record):
    """Scale metres to millimetres; discard unit and timestamp."""
    tag, value, unit, _timestamp = record
    scaled = value * 1000.0 if unit == 'm' else float(value)
    return tag, scaled

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

value 0 scales to 0.0. Non-'m' units pass through unchanged.

**Alternative approaches**

record[1] * 1000 works but hides which field is which - the very cost unpacking removes.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert summarise(('depth', 2.5, 'm', 1712000000.0)) == ('depth', 2500.0)
assert summarise(('time', 5.0, 's', 0)) == ('time', 5.0)

---

## Solution 5 — Build an index from coordinate tuples

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** hashable keys, dict from pairs, duplicate detection

**Explanation and reasoning**

The coordinate tuple is already hashable, so it becomes the key directly - converting it to a string would work but destroys the structure and makes later range queries impossible. Checking `in index` before assignment turns Python's silent dict overwrite-previous-value behaviour into an explicit failure.

In [ ]:
def build_index(cells):
    """Map coordinate tuples to labels, rejecting duplicates."""
    index = {}
    for coordinate, label in cells:
        if coordinate in index:
            raise ValueError(f'duplicate coordinate {coordinate!r}')
        index[coordinate] = label
    return index

**Complexity**

Time O(n) expected (hash lookups); space O(n).

**Edge cases and failure modes**

Empty input returns {}. The duplicate message contains the offending key.

**Alternative approaches**

dict(pairs) is one line but cannot detect duplicates - it overwrites silently.

**Tests — run the cell to verify the reference solution**

In [ ]:
idx = build_index([((0, 0), 'free'), ((1, 0), 'wall')])
assert idx[(0, 0)] == 'free'
try:
    build_index([((0, 0), 'a'), ((0, 0), 'b')])
    raise AssertionError('should raise')
except ValueError as exc:
    assert '(0, 0)' in str(exc)

---

## Solution 6 — Compare semantic versions as tuples

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** parsing, lexicographic comparison, tuple equality

**Explanation and reasoning**

Parsing to int tuples first is the whole fix: '(3, 9, 99)' as strings would compare '9' > '10' and give the wrong answer, while the int tuple compares 9 < 10 correctly. The try/except around int() re-raises with context so the exact bad part is visible in the traceback.

In [ ]:
def _parse(version):
    parts = version.split('.')
    if len(parts) != 3:
        raise ValueError(f'expected major.minor.patch, got {version!r}')
    try:
        return tuple(int(p) for p in parts)
    except ValueError as exc:
        raise ValueError(f'non-numeric part in {version!r}') from exc


def meets_minimum(version, minimum):
    """True when version >= minimum, compared numerically."""
    return _parse(version) >= _parse(minimum)

**Complexity**

Time O(len of both strings); space O(1) beyond the 3-tuples.

**Edge cases and failure modes**

'3.9.99' vs '3.10.0' returns True. Two-part or non-numeric versions raise ValueError.

**Alternative approaches**

packaging.version.parse is production-grade but hides the tuple lesson entirely.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert meets_minimum('3.12.1', '3.10.0') is True
assert meets_minimum('3.9.99', '3.10.0') is False
for bad in ('3.9', '3.x.0'):
    try:
        meets_minimum(bad, '1.0.0')
        raise AssertionError('should raise')
    except ValueError:
        pass

---

## Solution 7 — Make a nested structure hashable

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** deep freeze, recursion, hashability

**Explanation and reasoning**

Each container type is rebuilt as a tuple of recursively frozen parts, so hashability holds at every depth - the property `hash()` actually needs. Sorting dict items by a repr key gives a stable order even for mixed key types, which plain sorted() would refuse to compare.

In [ ]:
def freeze(value):
    """Recursively convert lists/dicts into hashable tuples."""
    if isinstance(value, dict):
        items = sorted(value.items(), key=lambda kv: repr(kv[0]))
        return tuple((k, freeze(v)) for k, v in items)
    if isinstance(value, list):
        return tuple(freeze(v) for v in value)
    if isinstance(value, (set, frozenset)):
        return tuple(sorted((freeze(v) for v in value), key=repr))
    return value   # scalars and tuples pass through

**Complexity**

Time O(n log n) from the sorts; space O(n).

**Edge cases and failure modes**

Scalars pass through unchanged. Empty dict/list become ().

**Alternative approaches**

json.dumps(..., sort_keys=True) yields a hashable string but loses type fidelity.

**Tests — run the cell to verify the reference solution**

In [ ]:
f = freeze({'b': [1, 2], 'a': 3})
assert f == (('a', 3), ('b', (1, 2)))
hash(f)
assert freeze(42) == 42
assert freeze({}) == ()

---

## Solution 8 — Swap-free rotation of field order

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** unpacking, cyclic shift, field rotation

**Explanation and reasoning**

Slicing works on tuples exactly as on lists and concatenation builds a new tuple, so the rotation is pure functional arithmetic with no mutation possible. Modulo first makes any k - negative or oversized - safe before the slices are taken.

In [ ]:
def rotate_fields(record, k):
    """Rotate an n-field tuple right by k."""
    if not record:
        return record
    k %= len(record)
    if k == 0:
        return record
    return record[-k:] + record[:-k]

**Complexity**

Time O(n); space O(n).

**Edge cases and failure modes**

Empty tuple returns itself. k = 0 returns the original object. Single field never changes.

**Alternative approaches**

Star-unpacking (last, *head = record) is elegant for k == 1 only; slicing generalises.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert rotate_fields(('a', 'b', 'c'), 1) == ('c', 'a', 'b')
assert rotate_fields(('a', 'b', 'c'), 0) == ('a', 'b', 'c')
assert rotate_fields(('a',), 5) == ('a',)

---

## Solution 9 — Group readings into (min, max, mean) summary tuples

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** aggregation, record emission, empty input

**Explanation and reasoning**

The empty guard runs before min/max, which would otherwise raise on an empty sequence - the documented zero tuple is a deliberate policy rather than a crash. Casting all three fields to float keeps the return type uniform even when a group holds ints.

In [ ]:
def summarise_groups(groups):
    """One (min, max, mean) tuple per group, zeros when empty."""
    summaries = []
    for group in groups:
        if not group:
            summaries.append((0.0, 0.0, 0.0))
            continue
        lo, hi = min(group), max(group)
        mean = float(sum(group)) / len(group)
        summaries.append((float(lo), float(hi), mean))
    return summaries

**Complexity**

Time O(total elements); space O(number of groups).

**Edge cases and failure modes**

Empty outer list returns []. Single-element groups give (x, x, x).

**Alternative approaches**

A comprehension with a helper function is flatter but splits the guard across scopes.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert summarise_groups([[1, 2, 3], []]) == [(1.0, 3.0, 2.0), (0.0, 0.0, 0.0)]
assert summarise_groups([]) == []
assert summarise_groups([[5]]) == [(5.0, 5.0, 5.0)]

---

## Solution 10 — Diff two record streams field by field

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** zip, field-wise diff, named reporting

**Explanation and reasoning**

zip(fields, a, b) aligns names with values in lockstep, so the report always carries the human-readable field name - something a plain `a != b` can never provide. The loop returns on the first difference, guaranteeing 'first' rather than 'any', and the explicit None return distinguishes equality from a missing result.

In [ ]:
def first_difference(a, b, fields):
    """(name, a, b) for the first mismatched field; None if equal."""
    if len(a) != len(b) or len(a) != len(fields):
        raise ValueError('records and field names must align')
    for name, va, vb in zip(fields, a, b):
        if va != vb:
            return (name, va, vb)
    return None

**Complexity**

Time O(k) for k fields (stops early); space O(1).

**Edge cases and failure modes**

Equal records return None. A last-field difference is still caught. Length mismatch raises.

**Alternative approaches**

next(((n, x, y) for n, x, y in zip(...) if x != y), None) is a one-liner but harder to test.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert first_difference((1.0, 'm'), (2.0, 'm'), ('value', 'unit')) == ('value', 1.0, 2.0)
assert first_difference((1, 'a'), (1, 'a'), ('n', 'u')) is None
assert first_difference((1, 0), (1, 1), ('n', 'k')) == ('k', 0, 1)

---

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.